In [6]:
import pandas as pd
import numpy as np
import os

# Verificar que los archivos existen
archivos = ['train.csv', 'test.csv', 'store.csv']
for f in archivos:
    path = f'../data/raw/{f}'
    size = os.path.getsize(path) / 1e6
    print(f"✓ {f}  →  {size:.1f} MB")
    

✓ train.csv  →  38.1 MB
✓ test.csv  →  1.4 MB
✓ store.csv  →  0.0 MB


In [7]:
# Cargar los datos
df_train = pd.read_csv('../data/raw/train.csv', 
                        parse_dates=['Date'],
                        dtype={'StateHoliday': str})
df_store = pd.read_csv('../data/raw/store.csv')

# Unir ambas tablas — siempre trabajaremos con el dataset combinado
df = df_train.merge(df_store, on='Store', how='left')

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"\nColumnas disponibles:")
print(list(df.columns))

Filas: 1,017,209
Columnas: 18

Columnas disponibles:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']


# 01 — Limpieza de Datos · Rossmann Store Sales

## Objetivo
Preparar el dataset crudo para el análisis exploratorio y modelado predictivo.

## Fuentes de datos
| Archivo | Filas | Descripción |
|---|---|---|
| train.csv | 1,017,209 | Ventas históricas por tienda y día |
| store.csv | 1,115 | Características de cada tienda |

## Transformaciones aplicadas
1. **Merge** de train + store por columna `Store`
2. **CompetitionDistance** — 2,642 nulos imputados con distancia máxima (75,860m)
3. **CompetitionOpenSince** — 323,348 nulos rellenados con 0
4. **Promo2Since + PromoInterval** — 508,031 nulos rellenados con 0 / 'None'
5. **Eliminación de días cerrados** — 172,817 filas removidas (Open=0)

## Resultado
Dataset limpio con **844,392 filas**, **0 nulos** y listo para EDA.

In [8]:
# Diagnóstico general del dataset
print("=" * 40)
print("TIPOS DE DATOS")
print("=" * 40)
print(df.dtypes)

print("\n" + "=" * 40)
print("VALORES NULOS POR COLUMNA")
print("=" * 40)
nulos = df.isnull().sum()
print(nulos[nulos > 0])

print("\n" + "=" * 40)
print("ESTADÍSTICAS BÁSICAS")
print("=" * 40)
print(df[['Sales', 'Customers', 'CompetitionDistance']].describe().round(2))

TIPOS DE DATOS
Store                                 int64
DayOfWeek                             int64
Date                         datetime64[us]
Sales                                 int64
Customers                             int64
Open                                  int64
Promo                                 int64
StateHoliday                            str
SchoolHoliday                         int64
StoreType                               str
Assortment                              str
CompetitionDistance                 float64
CompetitionOpenSinceMonth           float64
CompetitionOpenSinceYear            float64
Promo2                                int64
Promo2SinceWeek                     float64
Promo2SinceYear                     float64
PromoInterval                           str
dtype: object

VALORES NULOS POR COLUMNA
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2SinceWeek              508031
Promo2Si

In [11]:

# LIMPIEZA DE DATOS


df_clean = df.copy()

# 1. Imputar CompetitionDistance con la distancia máxima
df_clean['CompetitionDistance'] = df_clean['CompetitionDistance'].fillna(
    df_clean['CompetitionDistance'].max()
)

# 2. Rellenar nulos de competencia y promo con 0
cols_fill_zero = [
    'CompetitionOpenSinceMonth', 
    'CompetitionOpenSinceYear',
    'Promo2SinceWeek', 
    'Promo2SinceYear'
]
df_clean[cols_fill_zero] = df_clean[cols_fill_zero].fillna(0)

# 3. PromoInterval — rellenar con 'None'
df_clean['PromoInterval'] = df_clean['PromoInterval'].fillna('None')

# 4. Eliminar filas donde la tienda estaba cerrada (Open=0)
# No aportan información para predecir ventas
df_clean = df_clean[df_clean['Open'] == 1].copy()

# Verificar que no quedan nulos
print("Nulos restantes:", df_clean.isnull().sum().sum())
print(f"Filas después de limpieza: {df_clean.shape[0]:,}")
print(f"Filas eliminadas (tiendas cerradas): {df.shape[0] - df_clean.shape[0]:,}")

Nulos restantes: 0
Filas después de limpieza: 844,392
Filas eliminadas (tiendas cerradas): 172,817
